<a href="https://colab.research.google.com/github/maheshvlfm-coder/Train_ticketing_goal_programming/blob/main/Train_ticketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Train Ticketing Optimization Model
==================================
Objective: Maximize customer satisfaction while respecting capacity constraints

This model solves the train ticket confirmation problem by:
1. Prioritizing customers based on multiple criteria (priority score, fare willingness, booking time, etc.)
2. Ensuring capacity constraints are met for each train segment
3. Maximizing overall customer satisfaction and revenue potential

Author: AI Assistant
Date: August 2025
"""

import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta

class TrainTicketOptimizer:
    def __init__(self, train_capacity=200):
        self.train_capacity = train_capacity
        self.customers_df = None
        self.optimization_results = None

    def generate_customer_data(self, num_customers=100, num_stations=20, seed=42):
        """Generate synthetic customer booking data"""
        random.seed(seed)
        np.random.seed(seed)

        customers_data = []
        for i in range(num_customers):
            customer_id = f"CUST_{i+1:03d}"

            # Generate boarding and alighting stations
            boarding_station = random.randint(1, num_stations-1)
            alighting_station = random.randint(boarding_station+1, num_stations)
            journey_distance = alighting_station - boarding_station

            # Generate customer characteristics
            priority_score = round(random.uniform(1, 10), 2)
            fare_willingness = round(random.uniform(50, 500), 2)
            booking_time = datetime.now() - timedelta(days=random.randint(1, 30))

            passenger_types = ['Regular', 'Senior', 'Student', 'Business', 'Tourist']
            weights = [0.4, 0.15, 0.2, 0.15, 0.1]
            passenger_type = random.choices(passenger_types, weights=weights)[0]

            # Adjust priority based on passenger type
            if passenger_type == 'Senior':
                priority_score *= 1.3
            elif passenger_type == 'Student':
                priority_score *= 0.9
            elif passenger_type == 'Business':
                priority_score *= 1.2

            flexibility_score = round(random.uniform(1, 10), 2)
            group_size = random.choices([1, 2, 3, 4, 5], weights=[0.5, 0.25, 0.15, 0.08, 0.02])[0]

            customers_data.append({
                'customer_id': customer_id,
                'boarding_station': boarding_station,
                'alighting_station': alighting_station,
                'journey_distance': journey_distance,
                'priority_score': priority_score,
                'fare_willingness': fare_willingness,
                'booking_time': booking_time,
                'passenger_type': passenger_type,
                'flexibility_score': flexibility_score,
                'group_size': group_size
            })

        self.customers_df = pd.DataFrame(customers_data)
        return self.customers_df

    def calculate_utility_scores(self):
        """Calculate utility scores for customer prioritization"""
        if self.customers_df is None:
            raise ValueError("No customer data available. Call generate_customer_data() first.")

        df = self.customers_df.copy()

        # Normalize components
        df['priority_norm'] = df['priority_score'] / df['priority_score'].max()
        df['fare_norm'] = (df['fare_willingness'] - df['fare_willingness'].min()) / \
                         (df['fare_willingness'].max() - df['fare_willingness'].min())

        # Booking time advantage (earlier bookings get higher score)
        df['days_ago'] = df['booking_time'].apply(lambda x: (datetime.now() - x).days)
        df['booking_norm'] = df['days_ago'] / df['days_ago'].max()
        df['distance_norm'] = df['journey_distance'] / df['journey_distance'].max()

        # Group size penalty (smaller groups easier to accommodate)
        df['group_penalty'] = 1.0 / df['group_size']

        # Passenger type bonus
        type_bonus = {'Senior': 1.3, 'Business': 1.2, 'Regular': 1.0, 'Student': 0.9, 'Tourist': 1.0}
        df['type_bonus'] = df['passenger_type'].map(type_bonus)

        # Calculate composite utility score (weights can be adjusted based on business priorities)
        df['utility_score'] = (
            0.3 * df['priority_norm'] +     # Priority score weight
            0.25 * df['fare_norm'] +        # Fare willingness weight
            0.2 * df['booking_norm'] +      # Early booking weight
            0.15 * df['distance_norm'] +    # Journey distance weight
            0.1 * df['group_penalty']       # Group size weight
        ) * df['type_bonus']

        return df.sort_values('utility_score', ascending=False)

    def optimize_ticket_allocation(self):
        """
        Main optimization algorithm using greedy approach with constraint checking
        """
        if self.customers_df is None:
            raise ValueError("No customer data available. Call generate_customer_data() first.")

        # Calculate utility scores and sort customers
        customers_df_sorted = self.calculate_utility_scores()

        # Initialize segment usage tracking
        segment_usage = {}
        num_stations = self.customers_df['alighting_station'].max()
        for i in range(1, num_stations):
            segment_usage[(i, i+1)] = 0

        confirmed_customers = []
        rejected_customers = []
        allocation_details = []

        # Greedy allocation
        for idx, row in customers_df_sorted.iterrows():
            customer_id = row['customer_id']
            start = row['boarding_station']
            end = row['alighting_station']
            group_size = row['group_size']

            # Check capacity constraints for all segments this customer would use
            can_accommodate = True
            affected_segments = []

            for i in range(start, end):
                segment = (i, i+1)
                if segment in segment_usage:
                    if segment_usage[segment] + group_size > self.train_capacity:
                        can_accommodate = False
                        break
                    affected_segments.append(segment)

            if can_accommodate:
                # Confirm customer and update segment usage
                confirmed_customers.append(customer_id)
                for segment in affected_segments:
                    segment_usage[segment] += group_size

                allocation_details.append({
                    'customer_id': customer_id,
                    'status': 'Confirmed',
                    'utility_score': row['utility_score'],
                    'group_size': group_size,
                    'priority_score': row['priority_score'],
                    'fare_willingness': row['fare_willingness'],
                    'passenger_type': row['passenger_type'],
                    'segments_used': len(affected_segments),
                    'reason': 'Capacity available'
                })
            else:
                # Reject customer
                rejected_customers.append(customer_id)
                allocation_details.append({
                    'customer_id': customer_id,
                    'status': 'Rejected',
                    'utility_score': row['utility_score'],
                    'group_size': group_size,
                    'priority_score': row['priority_score'],
                    'fare_willingness': row['fare_willingness'],
                    'passenger_type': row['passenger_type'],
                    'segments_used': len(affected_segments),
                    'reason': 'Insufficient capacity'
                })

        # Store results
        self.optimization_results = {
            'confirmed_customers': confirmed_customers,
            'rejected_customers': rejected_customers,
            'allocation_details': pd.DataFrame(allocation_details),
            'segment_usage': segment_usage
        }

        return self.optimization_results

    def generate_report(self):
        """Generate comprehensive optimization report"""
        if self.optimization_results is None:
            raise ValueError("No optimization results available. Call optimize_ticket_allocation() first.")

        results = self.optimization_results
        confirmed_df = self.customers_df[self.customers_df['customer_id'].isin(results['confirmed_customers'])]
        rejected_df = self.customers_df[self.customers_df['customer_id'].isin(results['rejected_customers'])]

        report = f"""
TRAIN TICKET OPTIMIZATION REPORT
{'='*50}

BASIC STATISTICS:
- Total booking requests: {len(self.customers_df)}
- Confirmed bookings: {len(results['confirmed_customers'])} ({len(results['confirmed_customers'])/len(self.customers_df)*100:.1f}%)
- Rejected bookings: {len(results['rejected_customers'])} ({len(results['rejected_customers'])/len(self.customers_df)*100:.1f}%)

PASSENGER IMPACT:
- Confirmed passengers: {confirmed_df['group_size'].sum()}
- Rejected passengers: {rejected_df['group_size'].sum()}

REVENUE ANALYSIS:
- Revenue from confirmed bookings: ${confirmed_df['fare_willingness'].sum():.2f}
- Lost revenue from rejections: ${rejected_df['fare_willingness'].sum():.2f}
- Revenue capture rate: {confirmed_df['fare_willingness'].sum()/(confirmed_df['fare_willingness'].sum() + rejected_df['fare_willingness'].sum())*100:.1f}%

CUSTOMER TYPE BREAKDOWN (Confirmed / Total):
"""

        for ptype in self.customers_df['passenger_type'].unique():
            total_type = len(self.customers_df[self.customers_df['passenger_type'] == ptype])
            confirmed_type = len(confirmed_df[confirmed_df['passenger_type'] == ptype])
            report += f"- {ptype}: {confirmed_type}/{total_type} ({confirmed_type/total_type*100:.1f}%)\n"

        # Most utilized segments
        sorted_segments = sorted(results['segment_usage'].items(), key=lambda x: x[1], reverse=True)
        report += f"""
TOP 5 MOST UTILIZED SEGMENTS:
"""
        for i, (segment, usage) in enumerate(sorted_segments[:5]):
            utilization = usage / self.train_capacity * 100
            report += f"{i+1}. Segment {segment[0]} -> {segment[1]}: {usage}/{self.train_capacity} ({utilization:.1f}%)\n"

        print(report)
        return report

# Example usage:
if __name__ == "__main__":
    # Initialize optimizer
    optimizer = TrainTicketOptimizer(train_capacity=200)

    # Generate customer data
    customers = optimizer.generate_customer_data(num_customers=100, num_stations=20)
    print("Generated customer booking data")

    # Run optimization
    results = optimizer.optimize_ticket_allocation()
    print("Optimization completed")

    # Generate report
    optimizer.generate_report()

    # Save results
    optimizer.optimization_results['allocation_details'].to_csv('optimization_results.csv', index=False)
    print("Results saved to optimization_results.csv")

Generated customer booking data
Optimization completed

TRAIN TICKET OPTIMIZATION REPORT

BASIC STATISTICS:
- Total booking requests: 100
- Confirmed bookings: 100 (100.0%)
- Rejected bookings: 0 (0.0%)

PASSENGER IMPACT:
- Confirmed passengers: 195
- Rejected passengers: 0

REVENUE ANALYSIS:
- Revenue from confirmed bookings: $28442.86
- Lost revenue from rejections: $0.00
- Revenue capture rate: 100.0%

CUSTOMER TYPE BREAKDOWN (Confirmed / Total):
- Student: 23/23 (100.0%)
- Senior: 12/12 (100.0%)
- Business: 16/16 (100.0%)
- Regular: 36/36 (100.0%)
- Tourist: 13/13 (100.0%)

TOP 5 MOST UTILIZED SEGMENTS:
1. Segment 14 -> 15: 81/200 (40.5%)
2. Segment 12 -> 13: 74/200 (37.0%)
3. Segment 13 -> 14: 74/200 (37.0%)
4. Segment 15 -> 16: 71/200 (35.5%)
5. Segment 11 -> 12: 70/200 (35.0%)

Results saved to optimization_results.csv
